<a href="https://colab.research.google.com/github/psept77-star/RootCausePredictionMarketComplaint/blob/main/RootCausePredictionforMarketComplaint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#1. Import Liabraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

# load dataset
df = pd.read_csv('/content/sample_data/root_cause_dataset.csv')
df = df.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

In [6]:
# Align data anomalies/typos
df['Root_Cause'] = df['Root_Cause'].replace({'product damaged': 'Patient Misuse'})

#2. Define predictive test features and target
df['Feature_text'] = df['Complaint_Narrative'] + " " + df['Product_Name'] + " " + df['Severity']
x = df['Feature_text']
y = df['Root_Cause']

#Encode Multi class target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [8]:
#3. Train Test split
x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# 4. Feature extraction
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

#

In [15]:
#Model Selection
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest' : RandomForestClassifier(random_state=42),
    'Naive Bayes' : MultinomialNB(),
    'XGBoost' : XGBClassifier(random_state=42, eval_metric='mlogloss')
}

# Train, predict and evaluate all algorithms
for name, model in models.items():
  pipeline = Pipeline([
          ('vectorizer', tfidf),
          ('classifier', model)
      ])

  pipeline.fit(x_train, y_train)
  y_pred = pipeline.predict(x_test)

  acc = accuracy_score(y_test, y_pred)
  print(f"\n Model name: {name}")
  print(f"Accuracy: {acc:.4f}")
  print("Classification Matrix:")
  print(classification_report(y_test, y_pred, target_names=le.classes_))


 Model name: Logistic Regression
Accuracy: 1.0000
Classification Matrix:
                precision    recall  f1-score   support

  Distribution       1.00      1.00      1.00         5
 Manufacturing       1.00      1.00      1.00         5
     Packaging       1.00      1.00      1.00         5
Patient Misuse       1.00      1.00      1.00         5

      accuracy                           1.00        20
     macro avg       1.00      1.00      1.00        20
  weighted avg       1.00      1.00      1.00        20


 Model name: Random Forest
Accuracy: 0.9000
Classification Matrix:
                precision    recall  f1-score   support

  Distribution       1.00      0.60      0.75         5
 Manufacturing       0.71      1.00      0.83         5
     Packaging       1.00      1.00      1.00         5
Patient Misuse       1.00      1.00      1.00         5

      accuracy                           0.90        20
     macro avg       0.93      0.90      0.90        20
  weighted av

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# We will use Logistic Regression as it achieved 100% accuracy on our test set
# Let's rebuild and fit the best pipeline on the full training dataset first
best_pipeline = Pipeline([
    ('vectorizer', tfidf),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
best_pipeline.fit(x_train, y_train)

def predict_root_cause(product_name, complaint_narrative, severity="Major"):
    # Re-create the feature text string format used during model training
    feature_text = f"{complaint_narrative} {product_name} {severity}"

    # Predict class index
    pred_idx = best_pipeline.predict([feature_text])[0]

    # Decode index to actual class name
    pred_label = le.inverse_transform([pred_idx])[0]
    return pred_label

# Create Interactive UI Components
product_input = widgets.Text(placeholder='e.g., PainRelief Tablet', description='Product:')
complaint_input = widgets.Textarea(placeholder='e.g., Broken tablets found in blister pack', description='Complaint:')
severity_input = widgets.Dropdown(options=['Minor', 'Major', 'Critical'], value='Major', description='Severity:')
submit_btn = widgets.Button(description='Predict Root Cause', button_style='primary')
output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()
        prod = product_input.value.strip()
        complaint = complaint_input.value.strip()
        sev = severity_input.value

        if not prod or not complaint:
            print("Please provide both Product Name and Complaint Narrative.")
            return

        result = predict_root_cause(prod, complaint, sev)
        print(f"\nPredicted Root Cause: {result}")

submit_btn.on_click(on_predict_clicked)

# Display UI
display(product_input, complaint_input, severity_input, submit_btn, output_area)

Text(value='', description='Product:', placeholder='e.g., PainRelief Tablet')

Textarea(value='', description='Complaint:', placeholder='e.g., Broken tablets found in blister pack')

Dropdown(description='Severity:', index=1, options=('Minor', 'Major', 'Critical'), value='Major')

Button(button_style='primary', description='Predict Root Cause', style=ButtonStyle())

Output()